# Rank-8 minibatch Adam pilot

This is an exploratory comparison of full-batch and trial-minibatch Adam for the hierarchical MLMDP. The primary outcome is log likelihood per movement transition on the final m2 session, treated as a **pilot validation session**. It is not an untouched test set.

Run this notebook from the detached minibatch pilot worktree after the regression workflow finishes. Its data and selected rank-8 basis are read-only inputs from the original checkout. All outputs go to a new directory in this worktree. No NMF discovery or rank sweep runs here.

In [ ]:
import hashlib
import json
import sys
import tempfile
import uuid
from dataclasses import asdict
from datetime import UTC, datetime
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go
import torch

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "pyproject.toml").is_file():
    raise FileNotFoundError("Run from the pilot worktree or its notebook directory")
INPUT_ROOT = Path("/ceph/behrens/rudy/hierarchical_bigmaze")
DATA_ROOT = INPUT_ROOT / "external/GridMaze-mFC-ephys-DATA/data"
BASIS_ARTIFACT = INPUT_ROOT / "output/hierarchy_rank_discovery/k_08.json"
EXPECTED_PROFILE_SHA256 = (
    "7758ec2bf138ebc9cbc42ce61e1cc26e74abfb397213a9b24def2e8f2840815b"
)
CONFIG_PATH = PROJECT_ROOT / "configs/hierarchy_rank_validation_production.json"
OUTPUT_ROOT = PROJECT_ROOT / "output/minibatch_adam_pilot"
SAVED_RUN_DIR = None  # Set to a saved run directory to inspect it without refitting.
torch.set_num_threads(4)

sys.path.insert(0, str(PROJECT_ROOT / "src"))
from andrew_mlmdp import DoohanDataset, Environment  # noqa: E402
from andrew_mlmdp.hierarchy.minibatch_experiment import (  # noqa: E402
    PILOT_CHECKPOINTS,
    PILOT_EPOCHS,
    PILOT_LEARNING_RATES,
    PilotFitError,
    fit_minibatch_pilot,
)
from andrew_mlmdp.validation import (  # noqa: E402
    _data_fingerprint,
    _fitted_template,
    _initial_template,
    _maze_fingerprint,
    _strict_score,
    load_validation_config,
    source_code_fingerprint,
)

print("Pilot worktree:", PROJECT_ROOT)
print("Read-only input checkout:", INPUT_ROOT)
print("Output root:", OUTPUT_ROOT)

## Fixed input and split

Select the six m2 maze_1 sessions from the production configuration. Train on the first five chronological sessions and score the sixth. The dataset loader defines valid trials and extraction exclusions; this notebook does not filter trials by model score.

In [ ]:
config = load_validation_config(CONFIG_PATH)
assert config.dataset.subject_ids == ("m2",)
assert config.dataset.maze_name == "maze_1"
assert config.dataset.validation_mode == "chronological_holdout"
assert (
    config.dataset.training_session_count,
    config.dataset.validation_session_count,
) == (5, 1)

dataset = DoohanDataset.from_data_root(
    DATA_ROOT,
    subject_ids=config.dataset.subject_ids,
    start_date=config.dataset.start_date,
    end_date=config.dataset.end_date,
    maze_name=config.dataset.maze_name,
)
assert len(dataset.sessions) == 6
training_session_ids = tuple(session.session_id for session in dataset.sessions[:5])
validation_session_ids = (dataset.sessions[5].session_id,)
training_ids = set(training_session_ids)
validation_ids = set(validation_session_ids)
training_trials = tuple(
    trial for trial in dataset.trials if trial.session_id in training_ids
)
validation_trials = tuple(
    trial for trial in dataset.trials if trial.session_id in validation_ids
)
assert len(training_trials) == 326
assert len(validation_trials) == 78
assert not training_ids & validation_ids

print("Training sessions:", training_session_ids)
print("Pilot validation session:", validation_session_ids)
print(
    "Valid trials:",
    len(training_trials),
    "training;",
    len(validation_trials),
    "validation",
)
print("Data extraction exclusions:", len(dataset.exclusions))
for exclusion in dataset.exclusions:
    print("  ", exclusion)

In [ ]:
artifact = json.loads(BASIS_ARTIFACT.read_text(encoding="utf-8"))
assert artifact["status"] == "success" and artifact["k"] == 8
assert (
    artifact["configuration"]["discovery"] == config.normalized_payload()["discovery"]
)
assert artifact["compatibility"]["maze_sha256"] == _maze_fingerprint(dataset)
selected_discovery = artifact["discovery"]["selected_discovery"]
profiles = np.asarray(selected_discovery["profiles"], dtype=np.float64)
assert profiles.shape[1] == 8
profile_sha256 = hashlib.sha256(profiles.tobytes()).hexdigest()
assert profile_sha256 == selected_discovery["profile_sha256"]
assert profile_sha256 == EXPECTED_PROFILE_SHA256

environment = Environment(dataset.definition.maze)
initial_template, threshold_range, initial_values = _initial_template(
    environment,
    profiles,
    config,
    8,
    {trial.goal for trial in training_trials},
)
# Strict scoring reports an impossible required trial instead of dropping it.
initial_training_score = _strict_score(initial_template, training_trials)
initial_validation_score = _strict_score(initial_template, validation_trials)
print("Basis SHA-256:", profile_sha256)
print("Selected NMF seed:", artifact["discovery"]["selected_seed"])
print(
    "Initial training score:",
    initial_training_score["pooled_log_likelihood_per_transition"],
)
print(
    "Initial pilot validation score:",
    initial_validation_score["pooled_log_likelihood_per_transition"],
)

## Prespecified comparison

One full-batch run and three minibatch runs start from the same six parameter values and use 40 passes over the 326 training trials. Minibatches contain 32 whole trials, shuffled once per pass with seeds 0, 1, and 2. The last smaller batch is retained and weighted by its size. All runs use learning rates 0.05 for passes 1–20, 0.015 for 21–35, and 0.0045 for 36–40, with Adam state retained throughout. Scores are recorded after 0, 5, 10, 20, 30, 35, and 40 passes.

Final-iterate parameters are reported. Finite validation scores are measured at checkpoints but never choose parameters, learning rates, or an adaptive stopping time. A nonfinite required score fails that run visibly. This isolates the effect of stochastic batches under a common schedule; it does not reproduce the production plateau scheduler.

In [ ]:
def atomic_json(path, payload):
    path.parent.mkdir(parents=True, exist_ok=True)
    with tempfile.NamedTemporaryFile(
        mode="w", encoding="utf-8", dir=path.parent, suffix=".tmp", delete=False
    ) as handle:
        temporary = Path(handle.name)
        json.dump(payload, handle, indent=2, sort_keys=True, allow_nan=False)
    temporary.replace(path)


run_id = datetime.now(UTC).strftime("%Y%m%dT%H%M%SZ") + "_" + uuid.uuid4().hex[:8]
run_dir = OUTPUT_ROOT / run_id
run_dir.mkdir(parents=True, exist_ok=False)
source = source_code_fingerprint(PROJECT_ROOT, config_path=CONFIG_PATH)
specification = {
    "purpose": "exploratory m2 rank-8 minibatch Adam comparison",
    "worktree": str(PROJECT_ROOT),
    "input_checkout": str(INPUT_ROOT),
    "configuration": config.normalized_payload(),
    "source": {
        "git_head": source["git_head"],
        "content_sha256": source["content_sha256"],
    },
    "data_sha256": _data_fingerprint(dataset),
    "basis_artifact": str(BASIS_ARTIFACT),
    "basis_profile_sha256": profile_sha256,
    "basis_selected_seed": artifact["discovery"]["selected_seed"],
    "training_session_ids": training_session_ids,
    "validation_session_ids": validation_session_ids,
    "training_trial_keys": [
        (trial.session_id, trial.trial_id) for trial in training_trials
    ],
    "validation_trial_keys": [
        (trial.session_id, trial.trial_id) for trial in validation_trials
    ],
    "data_exclusions": [asdict(exclusion) for exclusion in dataset.exclusions],
    "initial_values": initial_values,
    "fitted_names": config.adam.fitted_names,
    "epochs": PILOT_EPOCHS,
    "learning_rates": PILOT_LEARNING_RATES,
    "checkpoint_epochs": PILOT_CHECKPOINTS,
    "runs": [
        {"label": "full", "batch_size": len(training_trials), "shuffle_seed": None},
        *[
            {"label": f"mini_seed_{seed}", "batch_size": 32, "shuffle_seed": seed}
            for seed in (0, 1, 2)
        ],
    ],
    "score_unit": "summed trial log likelihood / summed movement transitions",
    "selection": "preset final iterate after 40 passes",
    "validation_role": "exploratory model-comparison session",
}
atomic_json(run_dir / "specification.json", specification)
print("Saving pilot results to", run_dir, flush=True)

for run in specification["runs"]:
    label = run["label"]
    progress_path = run_dir / f"{label}.progress.json"

    def report_run_progress(progress):
        checkpoint = progress.checkpoint
        atomic_json(
            progress_path,
            {
                "status": "running",
                "run": run,
                "updated_at": datetime.now(UTC).isoformat(),
                "progress": asdict(progress),
            },
        )
        scores = ""
        if checkpoint is not None:
            training_score = checkpoint.training_log_likelihood_per_transition
            validation_score = checkpoint.validation_log_likelihood_per_transition
            scores = (
                f" | train={training_score:.6f} | validation={validation_score:.6f}"
            )
        print(
            f"{label} | pass {progress.passes}/{progress.total_passes}"
            f" | updates={progress.updates} | lr={progress.learning_rate:.4g}"
            f" | elapsed={progress.elapsed_seconds:.1f}s{scores}",
            flush=True,
        )

    print("Starting", label, flush=True)
    try:
        result = fit_minibatch_pilot(
            initial_template,
            training_trials,
            validation_trials,
            names=config.adam.fitted_names,
            batch_size=run["batch_size"],
            shuffle_seed=run["shuffle_seed"],
            progress_callback=report_run_progress,
        )
    except PilotFitError as error:
        failure = {
            "status": "failure",
            "type": type(error).__name__,
            "message": str(error),
            "run": run,
        }
        atomic_json(run_dir / f"{label}.json", failure)
        atomic_json(progress_path, failure)
        print(label, "failed:", error, flush=True)
        continue

    fitted_template = _fitted_template(
        environment, profiles, config, 8, result.final_values, initial_template
    )
    final_validation = _strict_score(fitted_template, validation_trials)
    assert (
        abs(
            final_validation["total_log_likelihood"]
            - result.checkpoints[-1].validation_total_log_likelihood
        )
        < 1e-7
    )
    success = {
        "status": "success",
        "run": run,
        "fit": asdict(result),
        "final_validation_trial_scores": final_validation["trial_scores"],
    }
    atomic_json(run_dir / f"{label}.json", success)
    atomic_json(
        progress_path,
        {
            "status": "success",
            "run": run,
            "completed_at": datetime.now(UTC).isoformat(),
            "progress": asdict(result.checkpoints[-1]),
        },
    )
    print(
        label,
        "final validation nats/transition =",
        result.checkpoints[-1].validation_log_likelihood_per_transition,
        flush=True,
    )

## Outcome

Compare runs at the same pass counts. The paired difference uses the same 78 validation trials and divides the difference in total log likelihood by their total movement transitions. A positive value favors minibatch Adam. The three seeds show sensitivity to trial order. Two or more positive final differences without failures warrant a broader six-fold experiment; this pilot alone is not a final generalization claim.

To inspect a saved run in a fresh kernel, set SAVED_RUN_DIR in the setup cell and execute the setup and Outcome cells. The fit cell can be skipped.

In [ ]:
results_dir = run_dir if SAVED_RUN_DIR is None else Path(SAVED_RUN_DIR)
saved_specification = json.loads(
    (results_dir / "specification.json").read_text(encoding="utf-8")
)
records = {}
for run in saved_specification["runs"]:
    label = run["label"]
    records[label] = json.loads(
        (results_dir / f"{label}.json").read_text(encoding="utf-8")
    )
if records["full"]["status"] != "success":
    raise RuntimeError("The matched full-batch reference failed")

rows = [
    {"run": label, **checkpoint}
    for label, payload in records.items()
    if payload["status"] == "success"
    for checkpoint in payload["fit"]["checkpoints"]
]
curves = pd.DataFrame(rows)
figure = go.Figure()
for label, group in curves.groupby("run", sort=False):
    figure.add_trace(
        go.Scatter(
            x=group["passes"],
            y=group["validation_log_likelihood_per_transition"],
            mode="lines+markers",
            name=label,
        )
    )
figure.update_layout(
    title="Pilot validation likelihood by training-data passes",
    xaxis_title="Training-data passes",
    yaxis_title="Validation log likelihood per movement transition (nats)",
    template="plotly_white",
)
figure.show()

full_scores = {
    (row["session_id"], row["trial_id"]): row
    for row in records["full"]["final_validation_trial_scores"]
}
comparison = []
for label in ("mini_seed_0", "mini_seed_1", "mini_seed_2"):
    payload = records[label]
    if payload["status"] != "success":
        comparison.append(
            {"run": label, "status": payload["status"], "reason": payload["message"]}
        )
        continue
    trial_deltas = []
    for score in payload["final_validation_trial_scores"]:
        key = (score["session_id"], score["trial_id"])
        baseline = full_scores[key]
        assert score["n_transitions"] == baseline["n_transitions"]
        trial_deltas.append(score["log_likelihood"] - baseline["log_likelihood"])
    total_transitions = sum(row["n_transitions"] for row in full_scores.values())
    delta = sum(trial_deltas) / total_transitions
    comparison.append(
        {
            "run": label,
            "status": "success",
            "delta_validation_nats_per_transition": delta,
            "trials_better": sum(value > 0 for value in trial_deltas),
            "trials_worse": sum(value < 0 for value in trial_deltas),
            "training_nats_per_transition": payload["fit"]["checkpoints"][-1][
                "training_log_likelihood_per_transition"
            ],
        }
    )
comparison_table = pd.DataFrame(comparison)
display(comparison_table)
finite_deltas = comparison_table["delta_validation_nats_per_transition"].dropna()
print("Median minibatch minus full-batch validation delta:", finite_deltas.median())
print("Min/max across seeds:", finite_deltas.min(), finite_deltas.max())
print("Seeds improving:", int((finite_deltas > 0).sum()), "of 3")
print(
    "Pilot criterion met:", len(finite_deltas) == 3 and (finite_deltas > 0).sum() >= 2
)